# OOD probe: AMPERSAND (Reddit CMV, Chakrabarty et al. 2019)Runs v4 on 150 balanced Reddit ChangeMyView sentences from AMPERSAND. Reports binary is-argumentative F1 and ternary role classification. See Section 6.1 of the paper.

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate
nohup python3 -u <<'PY' > eval_logs/aurc_v4.log 2>&1 &
import sys, json, random, time
sys.path.insert(0, '.')
import pandas as pd
import torch
from src.phase2.config import load_phase2_config
from src.phase2.student import build_student, _format_input

cfg = load_phase2_config('configs/phase2_beta_qwen1.5b_lora_v4.yaml')
cfg.student.max_input_len  = 2048
cfg.student.max_target_len = 1024
student = build_student(cfg.student)
student.load(cfg.student_output_dir)
print("loaded v4 (patched _parse_output)\n", flush=True)

data = pd.read_csv('phase2_data/raw/aurc/AURC/data/AURC_DATA.tsv', sep='\t')
stance = pd.read_csv('phase2_data/raw/aurc/AURC/data/AURC_SENTENCE_LEVEL_STANCE.tsv', sep='\t')
merged = data.merge(stance, on='sentence_hash', how='left')
merged = merged[merged['sentence'].notna() & (merged['sentence'].str.len() > 20)]
print(f"AURC merged: {len(merged)} rows, {merged['topic'].nunique()} topics\n", flush=True)

def is_arg(s): return str(s).lower() in ('pro', 'con')
merged['is_arg'] = merged['sentence_level_stance'].apply(is_arg)
print(f"gold argumentative rate: {merged['is_arg'].mean():.1%}\n", flush=True)

random.seed(42)
sample = []
for topic in sorted(merged['topic'].unique()):
    tdf = merged[merged['topic'] == topic]
    args    = tdf[tdf['is_arg']].sample(min(5, tdf['is_arg'].sum()),   random_state=42)
    nonargs = tdf[~tdf['is_arg']].sample(min(5, (~tdf['is_arg']).sum()), random_state=42)
    sample.extend(args.to_dict('records'))
    sample.extend(nonargs.to_dict('records'))
print(f"balanced sample: {len(sample)} sentences\n", flush=True)

tp=fp=fn=tn=0
per_topic={}
t0=time.time()

for idx, row in enumerate(sample, 1):
    topic = row['topic']; sentence = row['sentence']; gold_arg = row['is_arg']
    input_text = f"Topic: {topic}\n\n{sentence}"
    t = time.time()
    try:
        pred, _ = student.predict(input_text)
    except Exception as e:
        print(f"[{idx}/{len(sample)}] ERR: {e}", flush=True); continue
    dt = time.time() - t
    n_c = len(pred['claim_components']); n_p = len(pred['premise_components'])
    pred_arg = (n_c + n_p) > 0

    if pred_arg and gold_arg:     tp += 1; v='TP'
    elif pred_arg and not gold_arg: fp += 1; v='FP'
    elif not pred_arg and gold_arg: fn += 1; v='FN'
    else:                          tn += 1; v='TN'

    per_topic.setdefault(topic, {'tp':0,'fp':0,'fn':0,'tn':0})
    per_topic[topic][v.lower()] += 1

    if idx <= 8 or idx % 10 == 0:
        print(f"[{idx}/{len(sample)}] {topic[:18]:18s} gold={'ARG ' if gold_arg else 'NON '} "
              f"pred_c/p={n_c}/{n_p} {v} t={dt:.0f}s", flush=True)

def f1(tp,fp,fn):
    p = tp/max(tp+fp,1); r = tp/max(tp+fn,1)
    return 2*p*r/max(p+r,1e-6), p, r

overall_f1, P, R = f1(tp,fp,fn)
acc = (tp+tn) / max(tp+fp+fn+tn, 1)
print(f"\n=== AURC BINARY IS-ARG EVAL (n={tp+fp+fn+tn}) ===")
print(f"F1={overall_f1:.3f}  P={P:.3f}  R={R:.3f}  Acc={acc:.3f}")
print(f"TP={tp}  FP={fp}  FN={fn}  TN={tn}")
print(f"\n--- per topic ---")
print(f"{'topic':22s} {'F1':>6} {'Acc':>6}  TP FP FN TN")
for t,c in sorted(per_topic.items()):
    tf1,_,_ = f1(c['tp'],c['fp'],c['fn'])
    tacc = (c['tp']+c['tn'])/max(sum(c.values()),1)
    print(f"{t[:22]:22s} {tf1:.3f}  {tacc:.3f}   {c['tp']:2d} {c['fp']:2d} {c['fn']:2d} {c['tn']:2d}")
print(f"\nwall: {(time.time()-t0)/60:.1f} min")
PY
echo "PID: $!"
sleep 2
tail -5 eval_logs/aurc_v4.log

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate
nohup python3 -u <<'PY' > eval_logs/aurc_v4.log 2>&1 &
import sys, json, random, time
sys.path.insert(0, '.')
import pandas as pd
import torch
from src.phase2.config import load_phase2_config
from src.phase2.student import build_student, _format_input

cfg = load_phase2_config('configs/phase2_beta_qwen1.5b_lora_v4.yaml')
cfg.student.max_input_len  = 2048
cfg.student.max_target_len = 1024
student = build_student(cfg.student)
student.load(cfg.student_output_dir)
print("loaded v4 (patched _parse_output)\n", flush=True)

data = pd.read_csv('phase2_data/raw/aurc/AURC/data/AURC_DATA.tsv', sep='\t')
stance = pd.read_csv('phase2_data/raw/aurc/AURC/data/AURC_SENTENCE_LEVEL_STANCE.tsv', sep='\t')
merged = data.merge(stance, on='sentence_hash', how='left')

# Robust string filter — coerce to str first, then length check
merged['sentence'] = merged['sentence'].astype(str)
merged = merged[merged['sentence'].apply(lambda s: isinstance(s, str) and s not in ('nan', '') and len(s) > 20)]
print(f"AURC merged: {len(merged)} rows, {merged['topic'].nunique()} topics\n", flush=True)

def is_arg(s): return str(s).lower() in ('pro', 'con')
merged['is_arg'] = merged['sentence_level_stance'].apply(is_arg)
print(f"gold argumentative rate: {merged['is_arg'].mean():.1%}\n", flush=True)

random.seed(42)
sample = []
for topic in sorted(merged['topic'].unique()):
    tdf = merged[merged['topic'] == topic]
    n_a = int(tdf['is_arg'].sum()); n_n = int((~tdf['is_arg']).sum())
    args    = tdf[tdf['is_arg']].sample(min(5, n_a), random_state=42) if n_a else tdf.iloc[:0]
    nonargs = tdf[~tdf['is_arg']].sample(min(5, n_n), random_state=42) if n_n else tdf.iloc[:0]
    sample.extend(args.to_dict('records'))
    sample.extend(nonargs.to_dict('records'))
print(f"balanced sample: {len(sample)} sentences\n", flush=True)

tp=fp=fn=tn=0
per_topic={}
t0=time.time()

for idx, row in enumerate(sample, 1):
    topic = row['topic']; sentence = row['sentence']; gold_arg = row['is_arg']
    input_text = f"Topic: {topic}\n\n{sentence}"
    t = time.time()
    try:
        pred, _ = student.predict(input_text)
    except Exception as e:
        print(f"[{idx}/{len(sample)}] ERR: {e}", flush=True); continue
    dt = time.time() - t
    n_c = len(pred['claim_components']); n_p = len(pred['premise_components'])
    pred_arg = (n_c + n_p) > 0

    if pred_arg and gold_arg:     tp += 1; v='TP'
    elif pred_arg and not gold_arg: fp += 1; v='FP'
    elif not pred_arg and gold_arg: fn += 1; v='FN'
    else:                          tn += 1; v='TN'

    per_topic.setdefault(topic, {'tp':0,'fp':0,'fn':0,'tn':0})
    per_topic[topic][v.lower()] += 1

    if idx <= 8 or idx % 10 == 0:
        print(f"[{idx}/{len(sample)}] {topic[:18]:18s} gold={'ARG ' if gold_arg else 'NON '} "
              f"pred_c/p={n_c}/{n_p} {v} t={dt:.0f}s", flush=True)

def f1(tp,fp,fn):
    p = tp/max(tp+fp,1); r = tp/max(tp+fn,1)
    return 2*p*r/max(p+r,1e-6), p, r

overall_f1, P, R = f1(tp,fp,fn)
acc = (tp+tn) / max(tp+fp+fn+tn, 1)
print(f"\n=== AURC BINARY IS-ARG EVAL (n={tp+fp+fn+tn}) ===")
print(f"F1={overall_f1:.3f}  P={P:.3f}  R={R:.3f}  Acc={acc:.3f}")
print(f"TP={tp}  FP={fp}  FN={fn}  TN={tn}")
print(f"\n--- per topic ---")
print(f"{'topic':22s} {'F1':>6} {'Acc':>6}  TP FP FN TN")
for t,c in sorted(per_topic.items()):
    tf1,_,_ = f1(c['tp'],c['fp'],c['fn'])
    tacc = (c['tp']+c['tn'])/max(sum(c.values()),1)
    print(f"{t[:22]:22s} {tf1:.3f}  {tacc:.3f}   {c['tp']:2d} {c['fp']:2d} {c['fn']:2d} {c['tn']:2d}")
print(f"\nwall: {(time.time()-t0)/60:.1f} min")
PY
echo "PID: $!"
sleep 3
tail -10 eval_logs/aurc_v4.log

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate
python3 <<'PY'
import pandas as pd
df = pd.read_csv('phase2_data/raw/aurc/AURC/data/AURC_DATA.tsv', sep='\t')
print(f"total rows: {len(df)}")
print(f"columns: {list(df.columns)[:12]}")
print()

# Diagnose the 'sentence' column
sent_str = df['sentence'].astype(str)
print(f"sentence non-null: {df['sentence'].notna().sum()}")
print(f"sentence non-empty string: {(sent_str.str.strip() != '').sum()}")
print(f"sentence == 'nan' string: {(sent_str == 'nan').sum()}")
print(f"sentence length > 20: {(sent_str.str.len() > 20).sum()}")
print()

# Look at first 3 rows with actual text
non_empty = df[df['sentence'].astype(str).str.strip().str.len() > 5]
print(f"rows with non-empty sentence: {len(non_empty)}")
if len(non_empty):
    print("first 3 non-empty:")
    for _, r in non_empty.head(3).iterrows():
        print(f"  topic={r['topic']}  sentence={r['sentence'][:100]!r}")
print()

# Check merged_segments column
print("merged_segments samples (first 3 non-null):")
ms = df[df['merged_segments'].notna()].head(3)
for _, r in ms.iterrows():
    print(f"  {str(r['merged_segments'])[:200]!r}")
print()

# Try AMPERSAND fallback quickly
print("=== AMPERSAND fallback probe ===")
import os
amp = 'phase2_data/raw/AMPERSAND-EMNLP2019/claimtrain.tsv'
if os.path.exists(amp):
    df2 = pd.read_csv(amp, sep='\t', header=None, names=['sentence','label'])
    print(f"AMPERSAND train: {len(df2)} rows, labels: {df2['label'].value_counts().to_dict()}")
    print(f"first 5 rows:")
    for _, r in df2.head(5).iterrows():
        print(f"  label={r['label']}  '{str(r['sentence'])[:100]}'")
PY

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate
nohup python3 -u <<'PY' > eval_logs/ampersand_v4.log 2>&1 &
import sys, json, random, time
sys.path.insert(0, '.')
import pandas as pd
from src.phase2.config import load_phase2_config
from src.phase2.student import build_student

cfg = load_phase2_config('configs/phase2_beta_qwen1.5b_lora_v4.yaml')
cfg.student.max_input_len  = 2048
cfg.student.max_target_len = 1024
student = build_student(cfg.student)
student.load(cfg.student_output_dir)
print("loaded v4 (patched _parse_output)\n", flush=True)

df = pd.read_csv('phase2_data/raw/AMPERSAND-EMNLP2019/claimtrain.tsv',
                 sep='\t', header=None, names=['sentence','label'])
df['sentence'] = df['sentence'].astype(str)
df = df[df['sentence'].apply(lambda s: len(s.strip()) > 10 and s != 'nan')]
print(f"AMPERSAND clean: {len(df)} rows\n", flush=True)
print(f"label distribution: {df['label'].value_counts().to_dict()}\n", flush=True)

# Label convention (AMPERSAND paper): 0 = non-arg, 1 = premise, 2 = claim
LABEL_NAME = {0: 'non', 1: 'premise', 2: 'claim'}

random.seed(42)
per_class = 50
sample = []
for lab in [0, 1, 2]:
    subset = df[df['label'] == lab]
    take = subset.sample(min(per_class, len(subset)), random_state=42)
    sample.extend(take.to_dict('records'))
random.shuffle(sample)
print(f"balanced sample: {len(sample)} sentences ({per_class} per class)\n", flush=True)

def v4_label(n_c, n_p):
    """Map v4 output to one of {non, premise, claim}."""
    if n_c == 0 and n_p == 0: return 'non'
    if n_c > 0: return 'claim'   # claims dominate if both present
    return 'premise'

# ── Metrics accumulators ──
bin_tp = bin_fp = bin_fn = bin_tn = 0
tri = {c: {'tp':0,'fp':0,'fn':0} for c in ('non','premise','claim')}
t0 = time.time()

for idx, row in enumerate(sample, 1):
    sentence = row['sentence']; gold_label = int(row['label'])
    gold_name = LABEL_NAME[gold_label]
    t = time.time()
    try:
        pred, _ = student.predict(sentence)
    except Exception as e:
        print(f"[{idx}/{len(sample)}] ERR: {e}", flush=True); continue
    dt = time.time() - t
    n_c = len(pred['claim_components']); n_p = len(pred['premise_components'])
    pred_name = v4_label(n_c, n_p)

    # Binary is-arg (non vs premise/claim)
    gold_arg = gold_label in (1, 2)
    pred_arg = pred_name != 'non'
    if pred_arg and gold_arg:     bin_tp += 1
    elif pred_arg and not gold_arg: bin_fp += 1
    elif not pred_arg and gold_arg: bin_fn += 1
    else:                          bin_tn += 1

    # Ternary
    for c in ('non','premise','claim'):
        is_gold = (gold_name == c); is_pred = (pred_name == c)
        if is_gold and is_pred: tri[c]['tp'] += 1
        elif is_pred and not is_gold: tri[c]['fp'] += 1
        elif is_gold and not is_pred: tri[c]['fn'] += 1

    if idx <= 8 or idx % 15 == 0:
        print(f"[{idx}/{len(sample)}] gold={gold_name:7s} pred={pred_name:7s} "
              f"c/p={n_c}/{n_p} t={dt:.1f}s  '{sentence[:60]}'", flush=True)

def f1(tp,fp,fn):
    p=tp/max(tp+fp,1); r=tp/max(tp+fn,1)
    return 2*p*r/max(p+r,1e-6), p, r

bf1, bP, bR = f1(bin_tp, bin_fp, bin_fn)
bacc = (bin_tp + bin_tn) / max(bin_tp+bin_fp+bin_fn+bin_tn, 1)
print(f"\n=== BINARY IS-ARG (arg vs non) ===")
print(f"F1={bf1:.3f} P={bP:.3f} R={bR:.3f} Acc={bacc:.3f}")
print(f"TP={bin_tp} FP={bin_fp} FN={bin_fn} TN={bin_tn}")

print(f"\n=== TERNARY (non / premise / claim) ===")
f1s = []
for c in ('non','premise','claim'):
    tf1, tp_, tr_ = f1(tri[c]['tp'], tri[c]['fp'], tri[c]['fn'])
    f1s.append(tf1)
    print(f"  {c:8s} F1={tf1:.3f} P={tp_:.3f} R={tr_:.3f} "
          f"(tp={tri[c]['tp']} fp={tri[c]['fp']} fn={tri[c]['fn']})")
print(f"MACRO F1 = {sum(f1s)/len(f1s):.3f}")
print(f"\nwall: {(time.time()-t0)/60:.1f} min")
PY
echo "PID: $!"
sleep 3
tail -10 eval_logs/ampersand_v4.log